# Первый Colab run: шумоподавление русской речи

Этот ноутбук делает первый короткий end-to-end прогон:

- подключает Google Drive
- копирует проект в Colab runtime
- подготавливает маленький набор русской речи
- генерирует маленький synthetic noisy dataset
- обучает tiny STFT-модель
- сохраняет checkpoint **после каждой эпохи** и дублирует их на Google Drive
- сравнивает tiny model с `identity` baseline на `SI-SDR`

Важно:

- это **не** финальная модель, а быстрый диагностический прогон;
- notebook предполагает, что папка проекта уже лежит у вас на Google Drive.


## 1. Подключаем Google Drive и настраиваем пути

Перед запуском проверьте `PROJECT_DIR_ON_DRIVE`.

Пример:

- если проект лежит в `MyDrive/dl/project`, оставьте путь как есть;
- если у вас другая папка, просто поменяйте строку ниже.


In [ ]:
from google.colab import drive
from pathlib import Path
import os
import shutil
import subprocess

drive.mount('/content/drive')

PROJECT_DIR_ON_DRIVE = Path('/content/drive/MyDrive/dl/project')
RUN_NAME = 'first_colab_result'
RUNTIME_DIR = Path('/content/project')
DRIVE_RUN_DIR = PROJECT_DIR_ON_DRIVE / 'colab_runs' / RUN_NAME

assert PROJECT_DIR_ON_DRIVE.exists(), (
    f'Не найдена папка проекта на Google Drive: {PROJECT_DIR_ON_DRIVE}\n'
    'Исправьте PROJECT_DIR_ON_DRIVE и перезапустите ячейку.'
)

if RUNTIME_DIR.exists():
    shutil.rmtree(RUNTIME_DIR)

rsync_cmd = [
    'rsync', '-a', '--delete',
    '--exclude', '.venv',
    '--exclude', 'outputs',
    '--exclude', 'data',
    '--exclude', 'manifests',
    '--exclude', '.pytest_cache',
    '--exclude', '.ruff_cache',
    f'{PROJECT_DIR_ON_DRIVE}/',
    str(RUNTIME_DIR),
]
subprocess.run(rsync_cmd, check=True)

os.chdir(RUNTIME_DIR)
print('PROJECT_DIR_ON_DRIVE =', PROJECT_DIR_ON_DRIVE)
print('RUNTIME_DIR =', RUNTIME_DIR)
print('DRIVE_RUN_DIR =', DRIVE_RUN_DIR)
print('Файл notebook в runtime:', (RUNTIME_DIR / 'notebooks' / 'first_colab_run.ipynb').exists())


## 2. Устанавливаем зависимости

Здесь мы ставим:

- `uv`
- train stack
- data stack для подготовки маленького датасета


In [ ]:
!pip -q install uv
!uv sync --extra train --extra data --extra dev
!uv run noise-suppression env check


## 3. Готовим маленькие данные для первого результата

В этом прогоне берем:

- небольшой кусок `google/fleurs` для русской clean speech;
- маленький synthetic noise pool: `fan`, `keyboard`, `traffic`, `cafe-babble-like`.

Это не финальный датасет. Он нужен, чтобы получить **первый обучающийся результат** и проверить весь pipeline в Colab.


In [ ]:
!uv run python scripts/prepare_first_colab_dataset.py --output-root data/colab_seed --num-clean 120 --seed 42
!uv run noise-suppression manifest build data/colab_seed/clean manifests/colab_clean.jsonl --kind clean --speaker-depth 0
!uv run noise-suppression manifest build data/colab_seed/noise manifests/colab_noise.jsonl --kind noise
!uv run noise-suppression manifest summarize manifests/colab_clean.jsonl
!uv run noise-suppression manifest summarize manifests/colab_noise.jsonl


## 4. Создаем synthetic mixtures

Для первого Colab run берем умеренный размер:

- `1200` synthetic mixtures total;
- после split получится примерно `960 train / 240 val`.

Этого достаточно, чтобы увидеть, учится ли tiny model вообще.


In [ ]:
import json
from pathlib import Path

mix_config = {
    'seed': 42,
    'mixing': {
        'clean_manifest': '../manifests/colab_clean.jsonl',
        'noise_manifest': '../manifests/colab_noise.jsonl',
        'rir_manifest': None,
        'sample_rate': 16000,
        'num_examples': 1200,
        'min_duration_sec': 2.5,
        'max_duration_sec': 4.0,
        'snr_min_db': -2.0,
        'snr_max_db': 18.0,
        'focus_snr_min_db': 0.0,
        'focus_snr_max_db': 10.0,
        'focus_probability': 0.75,
        'reverb_probability': 0.0,
        'target_peak': 0.95,
    },
}

mix_config_path = Path('configs/colab_first_result.runtime.yaml')
mix_config_path.write_text(json.dumps(mix_config, ensure_ascii=False, indent=2), encoding='utf-8')
print(mix_config_path)
print(mix_config_path.read_text(encoding='utf-8'))


In [ ]:
!uv run noise-suppression mix plan configs/colab_first_result.runtime.yaml manifests/colab_mix_plan.jsonl
!uv run noise-suppression mix render manifests/colab_mix_plan.jsonl data/colab_rendered --overwrite
!uv run noise-suppression manifest split data/colab_rendered/rendered_colab_mix_plan.jsonl manifests/colab_train.jsonl manifests/colab_val.jsonl --val-ratio 0.2 --seed 42
!uv run noise-suppression baseline apply manifests/colab_val.jsonl outputs/colab_identity_val --mode identity
!uv run noise-suppression metrics si-sdr manifests/colab_val.jsonl outputs/colab_identity_val


## 5. Готовим train config с обязательным сохранением на Google Drive

Критически важный момент:

- `output_dir` живет в runtime для скорости;
- `checkpoint_mirror_dir` указывает на Google Drive;
- после **каждой эпохи** туда будут копироваться `epoch_XXX.pt`, `last.pt`, `best.pt`, `history.json`, `resolved_config.json`.


In [ ]:
train_config = {
    'seed': 42,
    'data': {
        'train_manifest': '../manifests/colab_train.jsonl',
        'val_manifest': '../manifests/colab_val.jsonl',
        'sample_rate': 16000,
        'segment_seconds': 3.0,
        'batch_size': 8,
        'num_workers': 2,
        'limit_train': None,
        'limit_val': None,
    },
    'model': {
        'n_fft': 512,
        'hop_length': 128,
        'win_length': 512,
        'hidden_channels': 24,
    },
    'training': {
        'epochs': 6,
        'learning_rate': 0.001,
        'waveform_loss_weight': 1.0,
        'magnitude_loss_weight': 0.5,
        'device': 'auto',
        'output_dir': '../outputs/colab_first_result',
        'checkpoint_mirror_dir': str(DRIVE_RUN_DIR),
        'save_every_epoch': True,
        'log_interval': 20,
    },
}

train_config_path = Path('configs/colab_train.runtime.yaml')
train_config_path.write_text(json.dumps(train_config, ensure_ascii=False, indent=2), encoding='utf-8')
print(train_config_path)
print(train_config_path.read_text(encoding='utf-8'))


In [ ]:
!uv run noise-suppression train fit configs/colab_train.runtime.yaml


## 6. Смотрим историю обучения и checkpoint-ы


In [ ]:
import json
import matplotlib.pyplot as plt

runtime_output_dir = Path('outputs/colab_first_result')
history = json.loads((runtime_output_dir / 'history.json').read_text(encoding='utf-8'))['history']

print('Runtime checkpoints:')
for path in sorted(runtime_output_dir.glob('*')):
    print('  ', path.name)

print('\nMirrored checkpoints on Google Drive:')
for path in sorted(DRIVE_RUN_DIR.glob('*')):
    print('  ', path.name)

epochs = [row['epoch'] for row in history]
train_loss = [row['train_loss'] for row in history]
val_loss = [row['val_loss'] for row in history]
val_sisdr = [row['val_si_sdr'] for row in history]

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(epochs, train_loss, marker='o', label='train_loss')
plt.plot(epochs, val_loss, marker='o', label='val_loss')
plt.xlabel('epoch')
plt.ylabel('loss')
plt.title('Loss curves')
plt.grid(True)
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(epochs, val_sisdr, marker='o', color='tab:green', label='val_si_sdr')
plt.xlabel('epoch')
plt.ylabel('SI-SDR')
plt.title('Validation SI-SDR')
plt.grid(True)
plt.legend()
plt.show()

history


## 7. Валидируем `best.pt` на val subset


In [ ]:
!uv run noise-suppression train infer outputs/colab_first_result/best.pt manifests/colab_val.jsonl outputs/colab_model_val
!uv run noise-suppression metrics si-sdr manifests/colab_val.jsonl outputs/colab_identity_val
!uv run noise-suppression metrics si-sdr manifests/colab_val.jsonl outputs/colab_model_val


## 8. Слушаем один пример

Это полезно, чтобы проверить не только метрику, но и характер артефактов.


In [ ]:
from IPython.display import Audio, display
import json

val_rows = [json.loads(line) for line in Path('manifests/colab_val.jsonl').read_text(encoding='utf-8').splitlines() if line.strip()]
sample = val_rows[0]
enhanced_path = Path('outputs/colab_model_val') / f"{sample['id']}.wav"

print('Sample id:', sample['id'])
print('Noisy:', sample['noisy_path'])
print('Clean:', sample['clean_path'])
print('Enhanced:', enhanced_path)

display(Audio(sample['noisy_path']))
display(Audio(sample['clean_path']))
display(Audio(str(enhanced_path)))


## 9. Что делать дальше

Если этот первый run прошел нормально, следующий разумный шаг:

1. увеличить число synthetic mixtures до `2000-4000`;
2. расширить noise pool;
3. добавить маленький real noisy eval set;
4. заменить `TinyMaskNet` на `FullSubNet-like` backbone;
5. сравнить результат с `DeepFilterNet`.

Checkpoint-ы уже лежат на Google Drive, так что даже при аварии runtime вы не потеряете прогресс после последней завершенной эпохи.
